In [ ]:
import os
import pandas as pd

## first set of fractions
fractions = [1.0, 0.1, 0.01, 0.001]

sheet_map = {
    1.0: "100%",
    0.1: "10%",
    0.01: "1%",
    0.001: "0%", 
}

def load_all_pfba_models(folder):
    all_models = {}

    for fname in os.listdir(folder):
        if not fname.endswith(".xlsx") or fname.startswith("~$"):
            continue

        model_id = fname.replace("_pfba.xlsx", "")
        fpath = os.path.join(folder, fname)

        try:
            xls = pd.ExcelFile(fpath, engine="openpyxl")
        except Exception as e:
            print(f"⚠️ Skipping file {fname}: {e}")
            continue

        model_data = {}
        for f in fractions:
            sheet = sheet_map[f]
            if sheet in xls.sheet_names:
                df = pd.read_excel(fpath, sheet_name=sheet, engine="openpyxl", header=0)

                # standardize column names (in case of whitespace)
                df.columns = [str(c).strip() for c in df.columns]

                model_data[f] = df
            else:
                model_data[f] = pd.DataFrame(columns=["Reactions","fluxes","names"])

        all_models[model_id] = model_data

    return all_models


In [ ]:
import os
import pandas as pd

## second set of fractions
fractions_2_5 = [0.05, 0.02]
sheet_map_2_5 = {0.05: "5%", 0.02: "2%"}

def load_pfba_2_5(folder):
    all_models = {}

    for fname in os.listdir(folder):
        if not fname.endswith(".xlsx") or fname.startswith("~$"):
            continue

        model_id = fname.replace("_pfba.xlsx", "")
        fpath = os.path.join(folder, fname)

        try:
            xls = pd.ExcelFile(fpath, engine="openpyxl")
        except Exception as e:
            print(f"⚠️ Skipping {fname}: {e}")
            continue

        model_data = {}
        for frac in fractions_2_5:
            sh = sheet_map_2_5[frac]
            if sh in xls.sheet_names:
                df = pd.read_excel(xls, sh)
                # normalize colnames
                df.columns = [str(c).strip() for c in df.columns]
                model_data[frac] = df
            else:
                model_data[frac] = pd.DataFrame(columns=["Reactions","fluxes","names"])

        all_models[model_id] = model_data

    return all_models

In [ ]:
import os
import pandas as pd

## third set of fractions
fractions_20_50 = [0.5, 0.2]
sheet_map_20_50 = {0.5: "50%", 0.2: "20%"}

def load_pfba_20_50(folder):
    all_models = {}

    for fname in os.listdir(folder):
        if not fname.endswith(".xlsx") or fname.startswith("~$"):
            continue

        model_id = fname.replace("_pfba.xlsx", "")
        fpath = os.path.join(folder, fname)

        try:
            xls = pd.ExcelFile(fpath, engine="openpyxl")
        except Exception as e:
            print(f"⚠️ Skipping {fname}: {e}")
            continue

        model_data = {}
        for frac in fractions_20_50:
            sh = sheet_map_20_50[frac]
            if sh in xls.sheet_names:
                df = pd.read_excel(xls, sh)
                # normalize colnames
                df.columns = [str(c).strip() for c in df.columns]
                model_data[frac] = df
            else:
                model_data[frac] = pd.DataFrame(columns=["Reactions","fluxes","names"])

        all_models[model_id] = model_data

    return all_models

In [ ]:
import os
import pandas as pd

## third set of fractions
fractions_02_05 = [0.005, 0.002]
sheet_map_02_05 = {0.005: "0.005", 0.002: "0.002"}

def load_pfba_02_05(folder):
    all_models = {}

    for fname in os.listdir(folder):
        if not fname.endswith(".xlsx") or fname.startswith("~$"):
            continue

        model_id = fname.replace("_pfba.xlsx", "")
        fpath = os.path.join(folder, fname)

        try:
            xls = pd.ExcelFile(fpath, engine="openpyxl")
        except Exception as e:
            print(f"⚠️ Skipping {fname}: {e}")
            continue

        model_data = {}
        for frac in fractions_02_05:
            sh = sheet_map_02_05[frac]
            if sh in xls.sheet_names:
                df = pd.read_excel(xls, sh)
                # normalize colnames
                df.columns = [str(c).strip() for c in df.columns]
                model_data[frac] = df
            else:
                model_data[frac] = pd.DataFrame(columns=["Reactions","fluxes","names"])

        all_models[model_id] = model_data

    return all_models

In [ ]:
pfba_data_1 = load_all_pfba_models("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba/")
pfba_data_2 = load_pfba_2_5("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba1/")
pfba_data_3 = load_pfba_20_50("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba2/")
pfba_data_4 = load_pfba_02_05("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba3/")

In [ ]:
import numpy as np

def ensure_groups_column(pfba_data, fill_value=np.nan):
    for model, frac_dict in pfba_data.items():
        for frac, df in frac_dict.items():
            if df is None or df.empty:
                continue

            if "groups" not in df.columns:
                df["groups"] = fill_value

    return pfba_data


In [ ]:
pfba_data_1 = ensure_groups_column(pfba_data_1)
pfba_data_2 = ensure_groups_column(pfba_data_2)
pfba_data_3 = ensure_groups_column(pfba_data_3)
pfba_data_4 = ensure_groups_column(pfba_data_4)

m = "ACH-000312"
print(pfba_data_2[m][0.02].shape)
print(pfba_data_3[m][0.2].columns)
print(pfba_data_1[m][0.001].head())


In [ ]:
def map_groups_from_excel(
    pfba_data,
    excel_df,
    rxn_col_excel="Reactions",
    group_col_excel="groups"
):
    # Build fast lookup dict: reaction_id -> group
    rxn_to_group = (
        excel_df[[rxn_col_excel, group_col_excel]]
        .dropna(subset=[rxn_col_excel])
        .set_index(rxn_col_excel)[group_col_excel]
        .to_dict()
    )

    for model, frac_dict in pfba_data.items():
        for frac, df in frac_dict.items():
            if df is None or df.empty:
                continue

            # Always overwrite / create groups cleanly
            df["groups"] = df["Reactions"].map(rxn_to_group)

    return pfba_data


In [ ]:
# Load your Excel mapping
excel_map = pd.read_excel("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Recon3D_groups.xlsx")

# Apply mapping
pfba_data_1 = map_groups_from_excel(
    pfba_data_1,
    excel_df=excel_map,
    rxn_col_excel="Reactions",
    group_col_excel="groups"
)



In [ ]:
# Load your Excel mapping
excel_map = pd.read_excel("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Recon3D_groups.xlsx")

# Apply mapping
pfba_data_2 = map_groups_from_excel(
    pfba_data_2,
    excel_df=excel_map,
    rxn_col_excel="Reactions",
    group_col_excel="groups"
)


In [ ]:
# Load your Excel mapping
excel_map = pd.read_excel("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Recon3D_groups.xlsx")

# Apply mapping
pfba_data_3 = map_groups_from_excel(
    pfba_data_3,
    excel_df=excel_map,
    rxn_col_excel="Reactions",
    group_col_excel="groups"
)


In [ ]:
# Load your Excel mapping
excel_map = pd.read_excel("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Recon3D_groups.xlsx")

# Apply mapping
pfba_data_4 = map_groups_from_excel(
    pfba_data_4,
    excel_df=excel_map,
    rxn_col_excel="Reactions",
    group_col_excel="groups"
)


In [ ]:
m = list(pfba_data_3.keys())[0]
f = sorted(pfba_data_3[m].keys())[0]
df = pfba_data_3[m][f]
df


In [ ]:
def add_ros_rss_groups_to_pfba(pfba_data):
    for model_id, model_data in pfba_data.items():
        for frac, df in model_data.items():
            if df is None or df.empty:
                continue

            # Only proceed if expected columns exist
            if "Reactions" not in df.columns or "groups" not in df.columns:
                continue

            df = df.copy()
            df.loc[df["Reactions"].str.endswith("demand", na=False), "groups"] = "RS_demand" 
            df.loc[df["Reactions"] == "H2O2_m_demand", "groups"] = "Mito_ROS_demand"
            df.loc[df["Reactions"] == "H2O2_x_demand", "groups"] = "Perox_ROS_demand"
            df.loc[df["Reactions"] == "oh_rad_c_demand", "groups"] = "Cyto_ROS_demand"
            df.loc[df["Reactions"] == "H2S_m_demand", "groups"] = "Mito_RSS_demand"
            df.loc[df["Reactions"] == "H2S_c_demand", "groups"] = "Cyto_RSS_demand"
            df.loc[df["Reactions"] == "HS_c_demand", "groups"] = "Cyto_RSS_demand"
            df.loc[df["Reactions"] == "Hypochlorous_c_demand", "groups"] = "Cyto_RHS_demand"
            df.loc[df["groups"] == "mem_transports", "groups"]="Transport, extracellular"
            df.loc[df["Reactions"] == "DM_h2o2[total]", "groups"] = "ROS_demand_total"
            df.loc[df["Reactions"] == "DM_oh_rad[total]", "groups"] = "ROS_demand_total"
            df.loc[df["Reactions"] == "DM_h2s[total]", "groups"] = "RSS_demand_total"
            df.loc[df["Reactions"] == "DM_HC00250[total]", "groups"] = "RSS_demand_total"
            df.loc[df["Reactions"] == "DM_CE4633[total]", "groups"] = "RHS_demand_total"
            df.loc[df["Reactions"].str.startswith("EX_", na=False), "groups"] = "Exchange"
            df.loc[df["Reactions"].str.startswith("sink_", na=False), "groups"] = "Sinks"
            # everything else that starts with DM_ becomes generic Demand
            mask = df["Reactions"].str.startswith("DM_", na=False) & ~df["Reactions"].isin(
                ["DM_h2o2[total]", "DM_oh_rad[total]", "DM_h2s[total]","DM_CE4633[total]","DM_HC00250[total]"]
            )
            df.loc[mask, "groups"] = "Demand"
            model_data[frac] = df  # write back
    return pfba_data


In [ ]:
pfba_data_1 = add_ros_rss_groups_to_pfba(pfba_data_1)
pfba_data_2 = add_ros_rss_groups_to_pfba(pfba_data_2)
pfba_data_3 = add_ros_rss_groups_to_pfba(pfba_data_3)
pfba_data_4 = add_ros_rss_groups_to_pfba(pfba_data_4)

In [ ]:
import pandas as pd

def add_rs_groups_from_excel(pfba_data, excel_path):
    rs_map = pd.read_excel(excel_path)

    # make a dict for fast lookup
    rs_dict = dict(zip(rs_map["Abbreviation"], rs_map["Subsystem"]))

    for model_id, model_data in pfba_data.items():
        for frac, df in model_data.items():
            if df is None or df.empty:
                continue
            if "Reactions" not in df.columns or "groups" not in df.columns:
                continue

            df = df.copy()
            mask = df["Reactions"].astype(str).str.startswith("RS_", na=False)

            # map group names where available
            df.loc[mask, "groups"] = df.loc[mask, "Reactions"].map(rs_dict).fillna(df.loc[mask, "groups"])

            model_data[frac] = df

    return pfba_data


In [ ]:
pfba_data_1 = add_rs_groups_from_excel(pfba_data_1, "/Users/subasrees/Downloads/Frontiers/Frontier_ccle/RSmodel_Recon3D1_2024_withGPRs_updated.xls")
pfba_data_2 = add_rs_groups_from_excel(pfba_data_2, "/Users/subasrees/Downloads/Frontiers/Frontier_ccle/RSmodel_Recon3D1_2024_withGPRs_updated.xls")
pfba_data_3 = add_rs_groups_from_excel(pfba_data_3, "/Users/subasrees/Downloads/Frontiers/Frontier_ccle/RSmodel_Recon3D1_2024_withGPRs_updated.xls")
pfba_data_4 = add_rs_groups_from_excel(pfba_data_4, "/Users/subasrees/Downloads/Frontiers/Frontier_ccle/RSmodel_Recon3D1_2024_withGPRs_updated.xls")

m = list(pfba_data_1.keys())[0]
f = sorted(pfba_data_3[m].keys())[0]
df = pfba_data_3[m][f]

df[df["Reactions"].str.startswith("RS_", na=False)]["groups"].value_counts().head(20)



In [ ]:
import pandas as pd

# 1️⃣ Load the Excel file containing the mapping
# Suppose your Excel file is named 'model_disease_mapping.xlsx' 
# and the sheet has columns: 'ModelID' and 'OncotreePrimaryDisease'

mapping_df = pd.read_excel('/Users/subasrees/Downloads/Frontiers/5_cancers/celllines.xlsx','Final')

In [ ]:
## STEP 1 — Build cancer type map from already-loaded DataFrame

# ---- Excel metadata file ----
METADATA_FILE       = '/Users/subasrees/Downloads/Frontiers/5_cancers/celllines.xlsx'
METADATA_SHEET      = 'Final'
COL_ACH_ID          = "ModelID"
COL_CANCER_TYPE     = "OncotreePrimaryDisease"
METADATA_HEADER_ROW = 1
meta = mapping_df[["ModelID", "OncotreePrimaryDisease"]].copy()
meta = meta[meta["ModelID"].str.startswith("ACH-", na=False)].copy()
meta["ModelID"]                = meta["ModelID"].str.strip()
meta["OncotreePrimaryDisease"] = meta["OncotreePrimaryDisease"].str.strip()
cancer_type_map = dict(zip(meta["ModelID"], meta["OncotreePrimaryDisease"]))

print(f"Loaded {len(cancer_type_map)} cell line → cancer type mappings")
print(f"Cancer types found: {sorted(meta['OncotreePrimaryDisease'].dropna().unique())}")
meta = meta[meta[COL_ACH_ID].str.startswith("ACH-", na=False)].copy()
meta[COL_ACH_ID]      = meta[COL_ACH_ID].str.strip()
meta[COL_CANCER_TYPE] = meta[COL_CANCER_TYPE].str.strip()
cancer_type_map = dict(zip(meta[COL_ACH_ID], meta[COL_CANCER_TYPE]))
print(f"  Loaded {len(cancer_type_map)} cell line → cancer type mappings")
print(f"  Cancer types: {sorted(meta[COL_CANCER_TYPE].dropna().unique())}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ---- Oxygen class assignment ----
OXYGEN_CLASS_MAP = {
    1.0:   "Normoxia",
    0.5:   "Normoxia",
    0.2:   "Hypoxia",
    0.1:   "Hypoxia",
    0.05:  "Hypoxia",
    0.02:  "Hypoxia",
    0.01:  "Extreme Hypoxia",
    0.005: "Extreme Hypoxia",
    0.002: "Extreme Hypoxia",
    0.001: "Extreme Hypoxia",
}
O2_ORDER = ["Normoxia", "Hypoxia", "Extreme Hypoxia"]

# ---- SHAP feature type 1: subsystem-level features ----
TOP_SHAP_GROUPS = [
    "Cholesterol metabolism",
    "Citric acid cycle",
    "Fatty acid oxidation",
    "Fatty acid synthesis",
    "Mito_RSS_demand",
    "Nucleotide metabolism",
    "ROS detoxification",
    "ROS_demand_total",
    'R_Glutaminolysis_vs_Total_TCA_input',
    'R_PPP_vs_Glycolysis__log_ratio',
    "Squalene and cholesterol synthesis",
    "Transport, endoplasmic reticular",
    "Transport, mitochondrial",
    "Tryptophan metabolism",
    "Valine, leucine, and isoleucine metabolism",
    "Vitamin A metabolism"
]
# Output DPI
DPI = 300

# Colour palette shared across all violin figures
PALETTE = {
    "Normoxia":        "#2166ac",
    "Hypoxia":         "#f4a582",
    "Extreme Hypoxia": "#d6604d",
}

In [ ]:
# STEP 2 — Merge 4 nested dicts into master DataFrame

DICT_FRACTION_MAP = {
    "pfba_data_1": ([1.0, 0.1, 0.01, 0.001], pfba_data_1),
    "pfba_data_2": ([0.05, 0.02],             pfba_data_2),
    "pfba_data_3": ([0.5, 0.2],               pfba_data_3),
    "pfba_data_4": ([0.005, 0.002],           pfba_data_4),
}

print("\nMerging 4 nested dicts into master DataFrame...")
records = []

for dict_name, (fractions, data_dict) in DICT_FRACTION_MAP.items():
    print(f"  Processing {dict_name} — fractions: {fractions}")
    for ach_id, frac_dict in tqdm(data_dict.items(), desc=f"  {dict_name}"):
        cancer_type = cancer_type_map.get(ach_id, "Unknown")
        for frac, df in frac_dict.items():
            if frac not in fractions:
                continue
            o2_class = OXYGEN_CLASS_MAP.get(frac, "Unknown")
            chunk = df[["Reactions", "fluxes", "names", "groups"]].copy()
            chunk.columns = ["reaction_id", "flux", "reaction_name", "subsystem"]
            chunk["cell_line"]   = ach_id
            chunk["cancer_type"] = cancer_type
            chunk["o2_fraction"] = frac
            chunk["o2_class"]    = o2_class
            records.append(chunk)

master = pd.concat(records, ignore_index=True)
master["subsystem"] = master["subsystem"].fillna("Unassigned")
master["flux"]      = pd.to_numeric(master["flux"], errors="coerce")
master["flux_abs"]  = master["flux"].abs()

print("Ratio features added to master")
master["o2_class"]  = pd.Categorical(
    master["o2_class"], categories=O2_ORDER, ordered=True
)
print(f"\nMaster DataFrame shape: {master.shape}")
print(f"  Unique cell lines:  {master['cell_line'].nunique()}")
print(f"  Unique reactions:   {master['reaction_id'].nunique()}")
print(f"  Unique subsystems:  {master['subsystem'].nunique()}")
print(f"  O2 class counts (samples):\n"
      f"{master.drop_duplicates(['cell_line','o2_fraction'])['o2_class'].value_counts()}")

# Verify SHAP groups exist in data
found_groups = [g for g in TOP_SHAP_GROUPS if g in master["subsystem"].unique()]
# Verify SHAP groups exist in data
found_groups   = [g for g in TOP_SHAP_GROUPS if g in master["subsystem"].unique()]
missing_groups = [g for g in TOP_SHAP_GROUPS if g not in master["subsystem"].unique()]
if missing_groups:
    print(f"\n  WARNING: These SHAP groups not found in data: {missing_groups}")
    print(f"  Available subsystems containing 'ROS': "
          f"{[s for s in master['subsystem'].unique() if 'ROS' in str(s)]}")

In [ ]:
from scipy.stats import variation
import os
import warnings
warnings.filterwarnings("ignore")
import matplotlib.gridspec as gridspec

In [ ]:
# remove oxidative phosphorylation from the analysis as it is constrained
found_groups=['Oxidative phosphorylation']
# aggregate subsystem flux per cell_line × o2_fraction 
group_agg_raw = (
    master[~master["subsystem"].isin(found_groups)]
    .groupby(["cell_line", "cancer_type", "o2_fraction", "o2_class", "subsystem"], observed=True)["flux_abs"]
    .sum()
    .reset_index()
    .rename(columns={"flux_abs": "group_flux", "subsystem": "feature"})
)

# Total flux per cell_line × o2_fraction
total_flux = (
    master
    .groupby(["cell_line", "o2_fraction"], observed=True)["flux_abs"]
    .sum()
    .reset_index(name="total_flux")
)

# Merge on both keys — no duplication
group_agg = group_agg_raw.merge(
    total_flux,
    on=["cell_line", "o2_fraction"],
    how="left"
)

# Normalise to percentage
group_agg["group_flux_pct"] = (
    group_agg["group_flux"] / (group_agg["total_flux"] + 1e-12)
) * 100

print("Rows per cell_line × o2_fraction (should be 1 per feature):")
print(
    group_agg.groupby(["cell_line", "o2_fraction", "feature"])
    .size()
    .value_counts()
)
print(f"\nMax group_flux_pct: {group_agg['group_flux_pct'].max():.2f}%")
# check
print(f"Transport mitochondrial normoxia median: "
      f"{group_agg[group_agg['feature']=='Transport, mitochondrial'].groupby('o2_class')['group_flux_pct'].median()}")

In [ ]:
# Rebuild cancer specific profiles from group_agg
cancer_profiles = (
    group_agg
    .groupby(["cancer_type", "feature", "o2_class"], observed=True)["group_flux_pct"]
    .median()
    .reset_index()
)

print(cancer_profiles.shape)
print(cancer_profiles.head())
print(f"Cancer types: {cancer_profiles['cancer_type'].nunique()}")
print(f"Features: {cancer_profiles['feature'].nunique()}")
print(f"O2 classes: {cancer_profiles['o2_class'].nunique()}")

In [ ]:
from scipy.stats import spearmanr

results_by_class = []

for o2_cls in O2_ORDER:
    # Filter to this oxygen class only
    cls_profile = cancer_profiles[cancer_profiles["o2_class"] == o2_cls]
    cls_cohort  = (
        cls_profile
        .groupby("feature", observed=True)["group_flux_pct"]
        .mean()
        .reset_index()
        .rename(columns={"group_flux_pct": "cohort_mean"})
    )

    for cancer in cls_profile["cancer_type"].unique():
        ct = cls_profile[cls_profile["cancer_type"] == cancer].merge(
            cls_cohort, on="feature"
        )
        if len(ct) < 5:
            continue
        rho, pval = spearmanr(ct["group_flux_pct"], ct["cohort_mean"])
        results_by_class.append({
            "cancer_type":  cancer,
            "o2_class":     o2_cls,
            "spearman_rho": round(rho, 4),
            "pvalue":       pval,
            "n_features":   len(ct),
        })

corr_class_df = pd.DataFrame(results_by_class)

# Print summary per oxygen class
for cls in O2_ORDER:
    subset = corr_class_df[corr_class_df["o2_class"] == cls]
    print(f"\n{cls}:")
    print(subset[["cancer_type","spearman_rho","pvalue","n_features"]]
          .sort_values("spearman_rho", ascending=False)
          .to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Prepare data
# Get consistent cancer type order — ranked by mean rho across all classes
mean_rho = (
    corr_class_df
    .groupby("cancer_type")["spearman_rho"]
    .mean()
    .sort_values(ascending=True)
)
cancer_order = list(mean_rho.index)

# Pivot for plotting
pivot = (
    corr_class_df
    .pivot(index="cancer_type", columns="o2_class", values="spearman_rho")
    .reindex(index=cancer_order, columns=O2_ORDER)
)

# Plot
fig, ax = plt.subplots(figsize=(9, 6))

y       = np.arange(len(cancer_order))
height  = 0.25
offsets = [-height, 0, height]

colors  = {
    "Extreme Hypoxia": "#d6604d",
    "Hypoxia":         "#f4a582",
    "Normoxia":        "#2166ac",
}

for i, cls in enumerate(O2_ORDER):
    vals  = pivot[cls].values
    bars  = ax.barh(
        y + offsets[i], vals,
        height=height * 0.9,
        color=colors[cls],
        edgecolor="white",
        linewidth=0.4,
        label=cls,
        zorder=2,
    )
    # Add rho labels
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(
                bar.get_width() + 0.003,
                bar.get_y() + bar.get_height() / 2,
                f"{val:.2f}",
                va="center", ha="left",
                fontsize=6.5, color="black"
            )

# Reference lines
ax.axvline(0.85, color="gray", linewidth=0.8,
           linestyle="--", alpha=0.6, zorder=1)
ax.axvline(1.0,  color="black", linewidth=0.4,
           linestyle=":", alpha=0.4, zorder=1)
ax.text(0.851, len(cancer_order) - 0.3, "ρ=0.85",
        fontsize=7, color="gray", va="bottom")

ax.set_yticks(y)
ax.set_xlim(0.58, 1.06)
ax.set_xlabel(
    "Spearman correlation with mean flux profile (ρ)",
    fontsize=13
)
ax.set_title(
    "Global metabolic reprogramming signatures\n"
    "across normoxia, hypoxia, and extreme hypoxia",
    fontsize=15, pad=10
)

# Legend — oxygen classes
legend_elements = [
    mpatches.Patch(facecolor=colors["Extreme Hypoxia"], label="Extreme hypoxia"),
    mpatches.Patch(facecolor=colors["Hypoxia"],         label="Hypoxia"),
    mpatches.Patch(facecolor=colors["Normoxia"],        label="Normoxia"),
    #mpatches.Patch(facecolor="white", edgecolor="none",
                   #label="* Hematological malignancy"),
]
ax.legend(handles=legend_elements, fontsize=12,
          bbox_to_anchor=(1.01, 0.5),
          loc="center left", framealpha=0.7)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="x", labelsize=13)
ax.grid(axis="x", linewidth=0.3, alpha=0.4, zorder=0)

plt.tight_layout()
#fig.savefig("/Users/subasrees/Desktop/Pediatric-cancer-GEM-ML/Additional/Upload/Figure_correlation_by_class.pdf", dpi=300, bbox_inches="tight")
#fig.savefig("/Users/subasrees/Desktop/Pediatric-cancer-GEM-ML/Additional/Upload/Figure_correlation_by_class.jpg", dpi=300, bbox_inches="tight")
print("Saved Figure_correlation_by_class.pdf/.png")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# SHAP values WITH dominant bin direction (▲ high regime, ▼ low regime)

SHAP_VALUES_WITH_BIN = {
    "Cholesterol metabolism":                     {"Normoxia": (-0.01, "▼"), "Hypoxia": (+0.01, "▲"), "Extreme Hypoxia": (+0.01, "▼")},
    "Citric acid cycle":                          {"Normoxia": (+0.02, "▲"), "Hypoxia": (-0.01, "▼"), "Extreme Hypoxia": (-0.02, "▲")},
    "Fatty acid oxidation":                       {"Normoxia": (+0.01, "▲"), "Hypoxia": (-0.003, "▲"), "Extreme Hypoxia": (+0.01, "▼")},
    "Fatty acid synthesis":                       {"Normoxia": (+0.01, "▲"), "Hypoxia": (+0.05, "▲"), "Extreme Hypoxia": (-0.05, "▲")},
    "Mito_RSS_demand":                            {"Normoxia": (-0.000, "▼"), "Hypoxia": (-0.02, "▲"), "Extreme Hypoxia": (+0.02, "▲")},
    "Nucleotide metabolism":                      {"Normoxia": (+0.03, "▲"), "Hypoxia": (-0.01, "▲"), "Extreme Hypoxia": (-0.02, "▲")},
    "ROS detoxification":                         {"Normoxia": (-0.04, "▼"), "Hypoxia": (+0.06, "▲"), "Extreme Hypoxia": (+0.11, "▼")},
    "ROS_demand_total":                           {"Normoxia": (+0.01, "▲"), "Hypoxia": (-0.04, "▼"), "Extreme Hypoxia": (+0.05, "▼")},
    'R_Glutaminolysis_vs_Total_TCA_input':        {"Normoxia": (+0.01, "▼"), "Hypoxia": (-0.01, "▲"), "Extreme Hypoxia": (+0.01, "▲")},
    'R_PPP_vs_Glycolysis__log_ratio':             {"Normoxia": (+0.02, "▼"), "Hypoxia": (-0.00, "▼"), "Extreme Hypoxia": (+0.02, "▲")},
    "Squalene and cholesterol synthesis":         {"Normoxia": (+0.03, "▼"), "Hypoxia": (-0.03, "▼"), "Extreme Hypoxia": (-0.00, "▼")},
    "Transport, endoplasmic reticular":           {"Normoxia": (+0.00, "▲"), "Hypoxia": (+0.03, "▼"), "Extreme Hypoxia": (-0.03, "▼")},
    "Transport, mitochondrial":                   {"Normoxia": (+0.07, "▲"), "Hypoxia": (-0.04, "▲"), "Extreme Hypoxia": (+0.04, "▼")},
    "Tryptophan metabolism":                      {"Normoxia": (-0.02, "▼"), "Hypoxia": (+0.01, "▼"), "Extreme Hypoxia": (+0.01, "▼")},
    "Valine, leucine, and isoleucine metabolism": {"Normoxia": (+0.01, "▲"), "Hypoxia": (-0.00, "▲"), "Extreme Hypoxia": (-0.01, "▲")},
    'Vitamin A metabolism':                       {"Normoxia": (-0.03, "▲"), "Hypoxia": (+0.02, "▲"), "Extreme Hypoxia": (+0.01, "▲")}
    #"Phenylalanine metabolism":                   {"Normoxia": (+0.01, "▼"), "Hypoxia": (+0.01, "▼"), "Extreme Hypoxia": (-0.02, "▼")},
    #"RSS_demand_total":                           {"Normoxia": (-0.00, "▲"), "Hypoxia": (-0.01, "▲"), "Extreme Hypoxia": (+0.01, "▲")},
    #"Mito_ROS_demand":                            {"Normoxia": (+0.03, "▲"), "Hypoxia": (-0.01, "▲"), "Extreme Hypoxia": (-0.01, "▲")},
}

O2_ORDER   = ["Extreme Hypoxia", "Hypoxia", "Normoxia"]
GROUP_ORDER = list(SHAP_VALUES_WITH_BIN.keys())

O2_CLASS_FRAC_MAP = {
    "Normoxia":        [1.0, 0.5],
    "Hypoxia":         [0.2, 0.1, 0.05, 0.02],
    "Extreme Hypoxia": [0.01, 0.005, 0.002, 0.001],
}

# Build SHAP matrix (values only) and annotation matrix (value + triangle)

shap_matrix  = pd.DataFrame(
    {grp: {cls: v for cls, (v, _) in cls_dict.items()}
     for grp, cls_dict in SHAP_VALUES_WITH_BIN.items()}
).T.reindex(index=GROUP_ORDER, columns=O2_ORDER)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

def plot_split_shap_from_dict(
    shap_dict,
    o2_order=["Normoxia", "Hypoxia", "Extreme Hypoxia"],
    figsize=(12, 9),
    savepath=None,
    fname="split_shap_heatmap",
    dpi=300,
):

    # Build matrices
    shap_matrix = pd.DataFrame({
        group: {cls: val for cls, (val, _) in cls_dict.items()}
        for group, cls_dict in shap_dict.items()
    }).T.reindex(columns=o2_order)

    bin_matrix = pd.DataFrame({
        group: {cls: sym for cls, (_, sym) in cls_dict.items()}
        for group, cls_dict in shap_dict.items()
    }).T.reindex(columns=o2_order)

    # numeric encoding for bin heatmap
    bin_map = {"▼": -1, "•": 0, "▲": 1}
    bin_numeric = bin_matrix.replace(bin_map)

    # Plot
    fig, axes = plt.subplots(
        1, 2,
        figsize=figsize,
        gridspec_kw={"width_ratios": [1.3, 1], "wspace": 0.15}
    )

    # Panel A — SHAP values
    ax0 = sns.heatmap(
        shap_matrix,
        ax=axes[0],
        cmap="RdBu_r",
        center=0,
        linewidths=0.5,
        linecolor="gray",
        annot=shap_matrix.round(2),
        fmt=".2f",
        annot_kws={"size": 12},
        cbar_kws={"label": "SHAP value (Δ probability)", "shrink": 0.7},
    )

    axes[0].set_title("A. SHAP values", fontsize=15)
    axes[0].set_xlabel("")
    axes[0].set_ylabel("")
    axes[0].tick_params(axis="x", rotation=30, labelsize=13)
    axes[0].tick_params(axis="y", labelsize=13)

    # Panel B — dominant bin
    cmap_bin = ListedColormap(["#66c2a5", "#fc8d62"])

    ax1 = sns.heatmap(
        bin_numeric,
        ax=axes[1],
        cmap=cmap_bin,
        vmin=-1, vmax=1,
        linewidths=0.5,
        linecolor="gray",
        annot=bin_matrix,
        fmt="",
        annot_kws={"size": 12, "weight": "bold"},
        cbar=False,
        yticklabels=False,
    )

    axes[1].set_title("B. Dominant effect bin", fontsize=15)
    axes[1].set_xlabel("")
    axes[1].set_ylabel("")
    axes[1].tick_params(axis="x", rotation=30, labelsize=13)
    axes[1].tick_params(axis="x", pad=5)
    # legend text
    axes[1].text(
        0.75, -0.02,
        "▼ low-feature value bin   ▲ high-feature value bin",
        transform=axes[1].transAxes,
        ha="center",
        fontsize=11
    )

    plt.tight_layout()
    savepath="/Users/subasrees/Desktop/Pediatric-cancer-GEM-ML/Additional/Upload/"
    #if savepath is not None:
        #plt.savefig(f"{savepath}/{fname}.pdf", dpi=dpi, bbox_inches="tight")
        #plt.savefig(f"{savepath}/{fname}.jpg", dpi=dpi, bbox_inches="tight")

    plt.show()

In [ ]:
plot_split_shap_from_dict(SHAP_VALUES_WITH_BIN)

In [ ]:
MANUAL_REACTION_SETS = {
    "Transport, mitochondrial": [
        "CITtam",    # Citrate transport — fatty acid synthesis substrate
        "PYRt2m",    # Pyruvate transport — TCA entry, HIF-1α/PDK1 suppressed
        "ASPGLUm",   # Aspartate-glutamate shuttle — redox balance
        "MALtm",     # Malate transport — malate-aspartate shuttle
        "ORNt3m",    # Ornithine transport — urea cycle/polyamine synthesis
    ],
    "Transport, endoplasmic reticular": [
        "CHSTEROLtrc",  # Cholesterol transport — oxygen-dependent sterol synthesis
        "BILIRUBtr",    # Lipid flip-flop — membrane remodelling under ER stress
        "CERT1rt",      # Ceramide transport — ER stress response
        "Rtotalter",    # Fatty acid intracellular transport
        "O2ter",        # Oxygen ER transport — oxidative folding capacity
    ],
}

In [ ]:
REACTION_SHORT_NAMES = {
    # ROS detoxification
    "SPODMx":          "Superoxide dismutase,peroxisome",
    "SPODMm":          "Superoxide dismutase,mitochondria",
    "SPODM":           "Superoxide dismutase,cytosol",
    "r0010":           "Hydrogen peroxide oxidoreductase",
    "CATm":            "Catalase,mitochondria",

    # Transport, mitochondrial
    "CITtam":          "Citrate import (rev)",
    "PYRt2m":          "Pyruvate import",
    "ASPGLUm":         "Asp-Glu shuttle",
    "MALtm":           "Malate import (rev)",
    "ORNt3m":          "Ornithine export (rev)",

    # Replace these entries in REACTION_SHORT_NAMES
    "HMR_2282": "Stearoyl-CoA 9-desaturase (C14)",
    "RE0583C":  "Octadecadienoyl-CoA reductase",
    "HMR_2227": "FAS — propanoyl-ACP condensation",
    "HMR_2228": "FAS — malonyl-ACP elongation",
    "HMR_2230": "FAS — dehydration step",

    # ROS_demand_total
    "DM_h2o2[total]":   "Hydrogen peroxide demand",
    "DM_oh_rad[total]": "Hydroxyl radical demand",

    # Squalene and cholesterol synthesis
    "DSREDUCr":        "Desmosterol reductase",
    "IPDDI":           "Isopentenyl-PP isomerase",
    "MEVK1c":          "Mevalonate kinase",
    "DPMVDc":          "Diphosphomevalonate decarboxylase",
    "PMEVKc":          "Phosphomevalonate kinase",

    # Transport, endoplasmic reticular
    "CHSTEROLtrc":     "Cholesterol export (rev)",
    "BILIRUBtr":       "Lipid flip-flop import (rev)",
    "CERT1rt":         "Ceramide import (rev)",
    "Rtotalter":       "Fatty acid import (rev)",
    "O2ter":           "Oxygen import (rev)",

    # Nucleotide metabolism
    "HMR_3966":        "Nucleoside-Triphosphate Diphosphatase",
    "HMR_7160":        "DNA-directed DNA polymerase",

    # Vitamin A metabolism
    "RADH":            "Retinal dehydrogenase",
    "RAI3":            "13-cis-retinoic acid isomerase",
    "RDH1":            "Retinol dehydrogenase (NADH)",
    "RADH2":           "Retinal dehydrogenase (NADPH)",
    "RDH1a":           "Retinol dehydrogenase (NADPH)",

    # Citric acid cycle
    "SUCOAS1m":        "Succinate-CoA ligase (GDP)",
    "ICDHyp":          "Isocitrate dehydrogenase (NADP⁺)",
    "SUCD1m":          "Succinate dehydrogenase",
    "SUCOASm":         "Succinate-CoA ligase (ADP)",
    "ICDHyrm":         "Isocitrate dehydrogenase (rev)",

    # Mito_RSS_demand
    "H2S_m_demand":    "Hydrogen Sulfide mitochondrial demand",

    # Tryptophan metabolism
    "HACD1m":          "3-OH-acyl-CoA dehydrogenase (mit)",
    "ECOAH1m":         "3-OH-acyl-CoA dehydratase (mit)",
    "HACD1x":          "3-OH-acyl-CoA dehydrogenase (perox)",
    "ECOAH1x":         "3-OH-acyl-CoA dehydratase (perox)",
    "2OXOADPTm":       "2-Oxoadipate shuttle",

    # PPP reactions
    "G6PDH1rer":       "Glucose 6-Phosphate Dehydrogenase",
    "PGLer":           "6-Phosphogluconolactonase",
    "GNDer":           "Phosphogluconate dehydrogenase",
}

In [ ]:
def get_rxn_name(master_df, rxn_id):
    if rxn_id in REACTION_SHORT_NAMES:
        return REACTION_SHORT_NAMES[rxn_id]
    names = master_df[master_df["reaction_id"] == rxn_id]["reaction_name"].dropna()
    if names.empty:
        return rxn_id
    name = str(names.iloc[0])
    return name[:42] + "..." if len(name) > 42 else name

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

# CONFIGURATION

DPI               = 300
O2_FRACTION_ORDER = [1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]
O2_LABELS         = ["100%","50%","20%","10%","5%","2%","1%","0.5%","0.2%","0.1%"]
O2_ORDER          = ["Normoxia", "Hypoxia", "Extreme Hypoxia"]
NORMOXIA_END      = 1.5
HYPOXIA_END       = 5.5

REACTION_COLORS = [
    "#1f77b4","#ff7f0e","#2ca02c",
    "#d62728","#9467bd","#8c564b",
    "#e377c2","#7f7f7f","#bcbd22",
]

# 12 features in 4×3 grid order
GRID_FEATURES = [
    # Row 1
    "ROS detoxification",
    "Transport, mitochondrial",
    "Fatty acid synthesis",
    # Row 2
    "ROS_demand_total",
    "Squalene and cholesterol synthesis",
    "Transport, endoplasmic reticular",
    # Row 3
    "Nucleotide metabolism",
    "Vitamin A metabolism",
    "Citric acid cycle",
    # Row 4
    "Mito_RSS_demand",
    "Tryptophan metabolism",
    "R_PPP_vs_Glycolysis",
]

# Manually curated reaction sets
MANUAL_REACTION_SETS = {
    "Transport, mitochondrial": [
        "CITtam",    # Citrate transport
        "PYRt2m",    # Pyruvate transport
        "ASPGLUm",   # Aspartate-glutamate shuttle
        "MALtm",     # Malate transport
        "ORNt3m",    # Ornithine transport
    ],
    "Transport, endoplasmic reticular": [
        "CHSTEROLtrc",  # Cholesterol transport
        "BILIRUBtr",    # Lipid flip-flop
        "CERT1rt",      # Ceramide transport
        "Rtotalter",    # Fatty acid transport
        "O2ter",        # Oxygen ER transport
    ],
}

# PPP and Glycolysis reaction sets for supplementary figure
PPP_REACTIONS        = ["G6PDH1er", "PGLer", "GNDer"]
GLYCOLYSIS_REACTIONS = ["HEX1", "PGI", "PFK", "FBA", "GAPD",
                         "PGK", "PGM", "ENO", "PYK"]
# Glutaminolysis/TCA reaction sets for supplementary figure
Glu_REACTIONS=         ["GLUDxm", "GLUNm", "ASPTA", "ALATA_L"]
TCA_REACTIONS=       ["PCm", "PDHm", "GLUDxm", "GLUNm","ASPTA", "ALATA_L"]
# GLOBAL STYLE — set once at top of script
plt.rcParams.update({
    "font.weight":        "bold",
    "axes.titleweight":   "bold",
    "axes.labelweight":   "bold",
    "figure.titleweight": "bold",
    "font.size":          12,      # base font size — scales everything up
    "axes.titlesize":     14,      # panel titles
    "axes.labelsize":     10,       # x/y axis labels
    "xtick.labelsize":    10,       # x tick labels
    "ytick.labelsize":    12,       # y tick labels
    "legend.fontsize":    12,       # legend text
    "lines.linewidth":    2.0,     # thicker lines — more visible
    "lines.markersize":   5,       # larger markers
})
#panel labels
panel_labels = [
    'A.', 'B.', 'C.', 'D.',
    'E.', 'F.', 'G.', 'H.',
    'I.', 'J.', 'K.', 'L.']
# HELPERS
def get_top_reactions_final(master_df, group_name, n=5):
    available = set(master_df["reaction_id"].unique())

    # Manual override
    if group_name in MANUAL_REACTION_SETS:
        found = [r for r in MANUAL_REACTION_SETS[group_name]
                 if r in available]
        if found:
            return found[:n]

    # CV with scale filter
    subset = master_df[master_df["subsystem"] == group_name].copy()
    if subset.empty:
        return []

    rxn_o2 = (
        subset.groupby(["reaction_id", "o2_fraction"])["flux"]
        .median()
        .unstack("o2_fraction")
        .reindex(columns=O2_FRACTION_ORDER)
        .dropna(how="all")
    )
    if rxn_o2.empty:
        return []

    rxn_o2["mean_abs"] = rxn_o2[O2_FRACTION_ORDER].abs().mean(axis=1)
    rxn_o2 = rxn_o2[rxn_o2["mean_abs"] > 1e-6].copy()
    if rxn_o2.empty:
        return []

    max_abs  = rxn_o2[O2_FRACTION_ORDER].abs().max(axis=1)
    p75_max  = max_abs.quantile(0.75)
    scale_ok = max_abs <= p75_max * 3
    rxn_o2   = rxn_o2[scale_ok].copy()
    if rxn_o2.empty:
        return []

    rxn_o2["cv"] = rxn_o2[O2_FRACTION_ORDER].std(axis=1) / (
        rxn_o2[O2_FRACTION_ORDER].abs().mean(axis=1) + 1e-12
    )

    return list(rxn_o2.sort_values("cv", ascending=False).head(n).index)


def add_background_shading(ax):
    ax.axvspan(-0.5,         NORMOXIA_END, alpha=0.06, color="#2166ac", zorder=0)
    ax.axvspan(NORMOXIA_END, HYPOXIA_END,  alpha=0.06, color="#f4a582", zorder=0)
    ax.axvspan(HYPOXIA_END,  9.5,          alpha=0.06, color="#d6604d", zorder=0)
    ax.axvline(NORMOXIA_END, color="gray", linewidth=0.5,
               linestyle="--", alpha=0.4, zorder=1)
    ax.axvline(HYPOXIA_END,  color="gray", linewidth=0.5,
               linestyle="--", alpha=0.4, zorder=1)
    ax.text(0.5,  1.02, "Normoxia",
            transform=ax.get_xaxis_transform(),
            ha="center", fontsize=8, color="#185FA5")
    ax.text(3.5,  1.02, "Hypoxia",
            transform=ax.get_xaxis_transform(),
            ha="center", fontsize=8, color="#993C1D")
    ax.text(7.5,  1.02, "Extreme hypoxia",
            transform=ax.get_xaxis_transform(),
            ha="center", fontsize=8, color="#d6604d")

def make_ax_bold(ax):
    for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] +
                  ax.get_xticklabels() + ax.get_yticklabels()):
        item.set_fontweight("bold")
    leg = ax.get_legend()
    if leg:
        for text in leg.get_texts():
            text.set_fontweight("bold")
def plot_reaction_trajectories_on_ax(ax, master_df, rxn_list,
                                      title, show_sd=False):
    add_background_shading(ax)
    x = list(range(len(O2_FRACTION_ORDER)))

    for c_idx, rxn_id in enumerate(rxn_list):
        subset = master_df[master_df["reaction_id"] == rxn_id]
        if subset.empty:
            continue

        traj = (subset.groupby("o2_fraction")["flux"]
                .median().reindex(O2_FRACTION_ORDER).values)
        sd   = (subset.groupby("o2_fraction")["flux"]
                .std().reindex(O2_FRACTION_ORDER).values)

        color = REACTION_COLORS[c_idx % len(REACTION_COLORS)]
        label = get_rxn_name(master_df, rxn_id)

        ax.plot(x, traj, color=color, linewidth=1.4,
                marker="o", markersize=2.5, label=label, zorder=2)

        if show_sd:
            valid = np.isfinite(traj) & np.isfinite(sd)
            if valid.any():
                ax.fill_between(
                    [xi for xi, v in zip(x, valid) if v],
                    [t - s for t, s, v in zip(traj, sd, valid) if v],
                    [t + s for t, s, v in zip(traj, sd, valid) if v],
                    alpha=0.07, color=color, zorder=1,
                )

    ax.set_xticks(x)
    ax.set_xticklabels(O2_LABELS, fontsize=10, rotation=45, ha="right")
    ax.set_xlabel("O₂ fraction", fontsize=10)
    ax.set_ylabel("Median flux\n(mmol/gDW/h)", fontsize=14)
    ax.tick_params(axis="y", labelsize=10)
    ax.axhline(0, color="gray", linewidth=0.3, linestyle=":", zorder=0)
    ax.set_title(title, fontsize=14, pad=14)
    ax.legend(fontsize=10.5, loc="best", framealpha=0.5,
              borderpad=1, labelspacing=0.5)

    make_ax_bold(ax)

def plot_ratio_trajectory_on_ax(ax, master_df, num_rxns, den_rxns,
                                  title, ylabel,
                                  color="#534AB7", show_sd=False):
    add_background_shading(ax)
    x = list(range(len(O2_FRACTION_ORDER)))

    num_df = (
        master_df[master_df["reaction_id"].isin(num_rxns)]
        .groupby(["cell_line", "o2_fraction"])["flux_abs"]
        .sum().reset_index().rename(columns={"flux_abs": "num"})
    )
    den_df = (
        master_df[master_df["reaction_id"].isin(den_rxns)]
        .groupby(["cell_line", "o2_fraction"])["flux_abs"]
        .sum().reset_index().rename(columns={"flux_abs": "den"})
    )
    ratio_df = num_df.merge(den_df, on=["cell_line", "o2_fraction"])
    ratio_df["log_ratio"] = np.log(
        (ratio_df["num"] + 1e-12) / (ratio_df["den"] + 1e-12)
    )

    medians = (ratio_df.groupby("o2_fraction")["log_ratio"]
               .median().reindex(O2_FRACTION_ORDER).values)
    sds     = (ratio_df.groupby("o2_fraction")["log_ratio"]
               .std().reindex(O2_FRACTION_ORDER).values)

    ax.plot(x, medians, color=color, linewidth=1.6,
            marker="o", markersize=3, zorder=2, label="Median log ratio")

    if show_sd:
        valid = np.isfinite(medians) & np.isfinite(sds)
        if valid.any():
            ax.fill_between(
                [xi for xi, v in zip(x, valid) if v],
                [m - s for m, s, v in zip(medians, sds, valid) if v],
                [m + s for m, s, v in zip(medians, sds, valid) if v],
                alpha=0.12, color=color, zorder=1, label="±1 SD"
            )

    ax.axhline(0, color="gray", linewidth=0.5,
               linestyle="--", zorder=0, label="_nolegend_")
    ax.set_xticks(x)
    ax.set_xticklabels(O2_LABELS, fontsize=10, rotation=45, ha="right")
    ax.set_xlabel("O₂ fraction", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=14)
    ax.tick_params(axis="y", labelsize=10)
    ax.set_title(title, fontsize=14, pad=14)
    ax.legend(fontsize=10.5, loc="best", framealpha=0.5)
    make_ax_bold(ax)

# STEP 1 — Print reaction selection summary

print("Reaction selection summary:")
for feature in GRID_FEATURES:
    if feature == "R_PPP_vs_Glycolysis":
        print(f"  R_PPP_vs_Glycolysis: ratio panel (PPP={PPP_REACTIONS})")
        continue
    rxns = get_top_reactions_final(master, feature, n=5)
    method = "manual" if feature in MANUAL_REACTION_SETS else "CV+scale"
    print(f"  {feature} [{method}]: {rxns}")

# STEP 2 — Figure 4: 12-panel combined trajectory grid (4 × 3)

print("\nGenerating Figure 4: 12-panel trajectory grid...")

fig4, axes4 = plt.subplots(
    4, 3,
    figsize=(8 * 3, 5.5 * 4), 
    gridspec_kw={"hspace": 0.55, "wspace": 0.40}
)
axes4_flat = axes4.flatten()

for panel_idx, feature in enumerate(GRID_FEATURES):
    
    ax = axes4_flat[panel_idx]

    # Add panel label — top left corner, outside plot area
    ax.text(
        -0.08, 1.08,                    # x, y in axes coordinates
        panel_labels[panel_idx],         # label text
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top", ha="left",
        color="black",
    )

    if feature == "R_PPP_vs_Glycolysis":
        plot_ratio_trajectory_on_ax(
            ax, master,
            num_rxns=PPP_REACTIONS,
            den_rxns=GLYCOLYSIS_REACTIONS,
            title="PPP / Glycolysis log ratio",
            ylabel="log(PPP / Glycolysis)",
            color="#534AB7",
            show_sd=False,
        )
    else:
        rxn_list = get_top_reactions_final(master, feature, n=5)
        if not rxn_list:
            axes4_flat[panel_idx].set_visible(False)
            continue
        plot_reaction_trajectories_on_ax(
            ax, master, rxn_list,
            title=feature,
            show_sd=False,
        )

fig4.suptitle(
    "Reaction-level flux trajectories for top SHAP-ranked metabolic features\n"
    "across the normoxic-to-extreme-hypoxic oxygen gradients",
    fontsize=16, y=0.95,fontweight="bold"
)
#fig4.savefig("/Users/subasrees/Desktop/Figures_upload/Figure_4_trajectory_grid.pdf", dpi=DPI, bbox_inches="tight")
#fig4.savefig("/Users/subasrees/Desktop/Figures_upload/Figure_4_trajectory_grid.jpg", dpi=DPI, bbox_inches="tight")
print("Saved Figure_4_trajectory_grid.pdf/.jpg")



In [ ]:
# STEP 3 — Supplementary: PPP/Glycolysis 3-panel

print("\nGenerating Supplementary Figure: PPP/Glycolysis 3-panel...")

fig_supp, axes_supp = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1 — PPP reaction trajectories
plot_reaction_trajectories_on_ax(
    axes_supp[0], master,
    rxn_list=PPP_REACTIONS,
    title="A.  Pentose phosphate pathway\nbranching from glycolysis",
    show_sd=False,
)

axes_supp[0].set_ylabel("Median flux\n(mmol/gDW/h))", fontsize=14)

# Panel 2 — Glycolysis top 5 by CV with scale filter
gly_cv = (
    master[master["reaction_id"].isin(GLYCOLYSIS_REACTIONS)]
    .groupby(["reaction_id", "o2_fraction"])["flux"]
    .median()
    .unstack("o2_fraction")
    .reindex(columns=O2_FRACTION_ORDER)
)
gly_cv["mean_abs"] = gly_cv[O2_FRACTION_ORDER].abs().mean(axis=1)
gly_cv["cv"]       = gly_cv[O2_FRACTION_ORDER].std(axis=1) / (
    gly_cv[O2_FRACTION_ORDER].abs().mean(axis=1) + 1e-12
)
max_abs  = gly_cv[O2_FRACTION_ORDER].abs().max(axis=1)
p75_max  = max_abs.quantile(0.75)
scale_ok = max_abs <= p75_max * 3
gly_top5 = list(
    gly_cv[scale_ok & (gly_cv["mean_abs"] > 1e-6)]
    .sort_values("cv", ascending=False)
    .head(5).index
)

plot_reaction_trajectories_on_ax(
    axes_supp[1], master,
    rxn_list=gly_top5,
    title="B.  Glycolysis",
    show_sd=False,
)
axes_supp[1].set_ylabel("Median flux\n(mmol/gDW/h)", fontsize=14)

# Panel 3 — PPP/Glycolysis log ratio with SD
plot_ratio_trajectory_on_ax(
    axes_supp[2], master,
    num_rxns=PPP_REACTIONS,
    den_rxns=GLYCOLYSIS_REACTIONS,
    title="C.  PPP / Glycolysis log ratio\n across oxygen classes",
    ylabel="log(PPP flux / Glycolysis flux)",
    color="#534AB7",
    show_sd=False,
)

fig_supp.suptitle(
    "PPP vs Glycolysis: component reaction trajectories and log ratio\n"
    "across the normoxic-to-extreme-hypoxic oxygen gradients",
    fontsize=16, y=0.98,fontweight="bold"
)
plt.tight_layout()
fig_supp.savefig("/Users/subasrees/Desktop/Figures_upload/Supp_Figure_PPP_Glycolysis.pdf", dpi=DPI, bbox_inches="tight")
fig_supp.savefig("/Users/subasrees/Desktop/Figures_upload/Supp_Figure_PPP_Glycolysis.jpg", dpi=DPI, bbox_inches="tight")
print("Saved Supp_Figure_PPP_Glycolysis.pdf/.jpg")

# STEP 4 — Supplementary: Glutaminolysis/TCA 3-panel

print("\nGenerating Supplementary Figure: Glutaminolysis/TCA 3-panel...")

fig_supp_2, axes_supp_2 = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1 — Glu reaction trajectories
plot_reaction_trajectories_on_ax(
    axes_supp_2[0], master,
    rxn_list=Glu_REACTIONS,
    title="A.  Glutaminolysis",
    show_sd=False,
)
axes_supp_2[0].set_ylabel("Median flux\n(mmol/gDW/h))", fontsize=14)

# Panel 2 — Glycolysis top 5 by CV with scale filter
plot_reaction_trajectories_on_ax(
    axes_supp_2[1], master,
    rxn_list=TCA_REACTIONS,
    title="B.  TCA driven \n by glutaminolysis",
    show_sd=False,
)
axes_supp_2[1].set_ylabel("Median flux\n(mmol/gDW/h)", fontsize=14)

# Panel 3 — Glutaminolysis driven TCA anaplerosis log ratio with SD
plot_ratio_trajectory_on_ax(
    axes_supp_2[2], master,
    num_rxns=Glu_REACTIONS,
    den_rxns=TCA_REACTIONS,
    title="C.  Glutaminolysis driven TCA log ratio\n across oxygen classes",
    ylabel="log(Glutaminolysis flux /\nTCA analplerosis flux)",
    color="#534AB7",
    show_sd=False,
)

fig_supp_2.suptitle(
    "Glutaminolysis driven TCA anaplerosis: component reaction trajectories and log ratio\n"
    "across the normoxic-to-extreme-hypoxic oxygen gradients",
    fontsize=16, y=0.98,fontweight="bold"
)
plt.tight_layout()
#fig_supp.savefig("/Users/subasrees/Desktop/Pediatric-cancer-GEM-ML/Additional/Upload/Supp_Figure_glu_tca.pdf", dpi=DPI, bbox_inches="tight")
#fig_supp.savefig("/Users/subasrees/Desktop/Pediatric-cancer-GEM-ML/Additional/Upload/Supp_Figure_glu_tca.jpg", dpi=DPI, bbox_inches="tight")
print("Saved Supp_Figure_Glutaminolysis_TCA.pdf/.jpg")